# 5 · Score a client list → ranked leads

Turns a client's company list into a ranked lead list. This is the **deployment** notebook:
`3_Flat_table.ipynb` builds the *training* table (features at a past `ASOF`, plus a label);
this builds the *scoring* table (features as of **today**, no label) and applies the saved model.

## What the client sends

One CSV — see `client/input/client_companies_TEMPLATE.csv`:

| column | required | purpose |
|---|---|---|
| `company_number` | yes | the only reliable key — names do not match reliably |
| `company_name` | no | for their own checking; ignored here |
| `relationship` | yes | `customer` → excluded from leads · `prospect` → scored |

Everything else — sector, region, age, charge history — comes from Companies House.

## The rule that keeps this honest

**The feature definitions here must match `3_Flat_table.ipynb` exactly.** The definitions *are*
the model: if `yrs_since_nonlloyds_chg` is computed differently at scoring time than at training
time, the coefficients are being applied to a different quantity and the scores are meaningless
without anything erroring. Section 3 is copied from that notebook deliberately, and section 4
checks the built columns against the model manifest before predicting.

In [1]:
# --- config -----------------------------------------------------------------
import sys, json, shutil, datetime as dt
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError(f"Cannot find paths.py in any parent of {Path.cwd()}")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from paths import (COMPANIES_CSV, CHARGES_CSV, CH_DATA, MEDIA_INDEX,
                   MODEL_FILE, MODEL_MANIFEST, CLIENT_INPUT, CLIENT_OUTPUT)

# ============================ SETTINGS ============================
# Every CSV sitting directly in client/input/. The glob is NON-recursive on
# purpose: subfolders (Template/, Archive/) are skipped, so the shipped template
# and any test fixture cannot leak into a real run just by being nearby.
CLIENT_FILES = sorted(CLIENT_INPUT.glob("*.csv"))

# Stop rather than quietly hand back a partial lead list. Stage 0 of
# 1_CompaniesHouse ingests these same files, so by the time you run this notebook
# every company should already be held -- a non-zero gap means the pull has not
# been run yet, not a normal state to work around.
REQUIRE_FULL_COVERAGE = True

# ASOF = today. Every feature is measured here, and unlike the training build
# there is no label: every conversion is still in the future.
ASOF = pd.Timestamp.today().normalize()

TOP_K = 100          # how many leads to hand over (also the Precision@K in the report)

# Soft re-rank on sector x region media volume. Evidence (see the ablation):
# sector-only vol_z showed NO signal once seasonality was controlled; sector x
# region did, but only on cells with real region data and NOT monotonically.
# So: keep the weight low, use real cells only, and treat this as a tie-breaker
# rather than a performance claim. GATE_WEIGHT = 0 disables it entirely.
USE_GATE     = True
GATE_WEIGHT  = 0.15
# ==================================================================
RUN_DIR = CLIENT_OUTPUT / dt.date.today().isoformat()
RUN_DIR.mkdir(parents=True, exist_ok=True)

if not CLIENT_FILES:
    raise FileNotFoundError(
        f"no CSVs directly in {CLIENT_INPUT} "
        "(files inside Template/ or Archive/ are deliberately ignored)")
print(f"client files ({len(CLIENT_FILES)}):")
for _f in CLIENT_FILES:
    print(f"   {_f.name}")
print(f"coverage    : {'must be complete' if REQUIRE_FULL_COVERAGE else 'partial allowed'}")
print(f"ASOF        : {ASOF.date()}  (features measured here, no label)")
print(f"output dir  : {RUN_DIR}")

client files (1):
   30k_sme_for_mock_up.csv
coverage    : must be complete
ASOF        : 2026-08-23  (features measured here, no label)
output dir  : /Users/natchalin_/Projects/final_project/Lloyds/client/output/2026-08-23


## 1 · Read and repair the client file

**Leading zeros are the failure mode to expect.** 28% of UK company numbers start with `0`, and
Excel silently strips them (`05914136` → `5914136`). Since UK company numbers are always 8
characters, `zfill(8)` repairs that automatically — so be forgiving on input and strict on
reporting, rather than sending the client away to fix their spreadsheet.

Alphanumeric numbers (`SC…` Scotland, `NI…` Northern Ireland) are already 8 characters, so
`zfill` leaves them untouched.

In [2]:
frames = []
for f in CLIENT_FILES:
    one = pd.read_csv(f, dtype=str)

    # Excel exports routinely carry thousands of trailing empty rows. Drop them
    # BEFORE anything else -- otherwise they become "00000NAN" and get reported
    # as companies that could not be found, which is nonsense.
    blank = one.isna().all(axis=1).sum()
    one = one.dropna(how="all")

    cols = {c.lower().strip(): c for c in one.columns}
    if "company_number" not in cols:
        raise ValueError(f"{f.name}: no 'company_number' column. Got {list(one.columns)}")
    one = one[one[cols["company_number"]].notna()]

    frames.append(pd.DataFrame({
        "com_num":      one[cols["company_number"]].astype(str).str.strip().str.upper(),
        "client_name":  one[cols["company_name"]] if "company_name" in cols else pd.NA,
        "relationship": (one[cols["relationship"]].astype(str).str.strip().str.lower()
                         if "relationship" in cols else "prospect"),
        "_source_file": f.name,
    }))
    print(f"{f.name}: {len(one):,} usable rows" + (f"  ({blank:,} blank rows dropped)" if blank else ""))

client = pd.concat(frames, ignore_index=True)
n_raw = len(client)

# --- repair Excel damage: pad to the canonical 8 characters ---
short = client["com_num"].str.len() < 8
client["com_num"] = client["com_num"].str.zfill(8)

bad_rel = ~client["relationship"].isin(["customer", "prospect"])
if bad_rel.any():
    print(f"WARNING: {int(bad_rel.sum()):,} rows have an unrecognised 'relationship' "
          f"({sorted(client.loc[bad_rel, 'relationship'].unique())[:5]}) -> treated as prospect")
    client.loc[bad_rel, "relationship"] = "prospect"

# A company can appear in two files with CONFLICTING relationships. Resolve it
# deterministically in the safe direction: 'customer' wins, because pitching to
# an existing customer is the worse error. ('customer' sorts before 'prospect'.)
conflicts = int((client.groupby("com_num")["relationship"].nunique() > 1).sum())
if conflicts:
    print(f"WARNING: {conflicts:,} companies appear with conflicting relationship "
          f"across files -> resolved to 'customer'")

dupes = int(client.duplicated("com_num").sum())
client = (client.sort_values("relationship")
                .drop_duplicates("com_num", keep="first")
                .reset_index(drop=True))

if not (client["relationship"] == "customer").any():
    print("\nNOTE: no rows are marked 'customer'. Nothing will be excluded, so the "
          "lead list may contain companies the client already banks with.")

print(f"\nrows received      : {n_raw:,}")
print(f"  leading-zero repaired : {int(short.sum()):,}")
print(f"  duplicates dropped    : {dupes:,}")
print(f"unique companies   : {len(client):,}")
print(client["relationship"].value_counts().to_string())

30k_sme_for_mock_up.csv: 99 usable rows

NOTE: no rows are marked 'customer'. Nothing will be excluded, so the lead list may contain companies the client already banks with.

rows received      : 99
  leading-zero repaired : 48
  duplicates dropped    : 7
unique companies   : 92
relationship
prospect    92


## 2 · Coverage check

**This is a safety net, not a workflow step.** Stage 0 of `1_CompaniesHouse.ipynb` reads the same
`client/input/*.csv` files and queues their company numbers for the pull, so by the time you run
this notebook every company should already be in `companies.csv`.

A non-zero gap here means the pull has not been run yet. `REQUIRE_FULL_COVERAGE = True` stops the
notebook rather than quietly producing a lead list that covers only part of the client's request —
a partial answer that looks complete is worse than a clear halt.

The correct order is:

> **1_CompaniesHouse** (Stage 0 → 1 → 3 → 4b) → **2_GDELT** (Part 5) → **5_score**

Set the flag to `False` only when you deliberately want to score just the companies you hold.

In [3]:
companies = pd.read_csv(COMPANIES_CSV, dtype=str, low_memory=False)
companies["com_num"] = companies["com_num"].astype(str).str.strip().str.upper().str.zfill(8)

known   = set(companies["com_num"])
missing = sorted(set(client["com_num"]) - known)

print(f"on the client list      : {len(client):,}")
print(f"already in companies.csv: {len(client) - len(missing):,}")
print(f"NOT held yet            : {len(missing):,}")

if missing:
    pd.Series(missing, name="com_num").to_csv(RUN_DIR / "awaiting_pull.csv", index=False)
    msg = (f"\n{len(missing):,} of {len(client):,} companies are not held yet "
           f"(listed in {RUN_DIR.name}/awaiting_pull.csv).\n"
           "Run 1_CompaniesHouse.ipynb (Stage 0 ingests client/input/, then Stages 1, 3, 4b), "
           "then 2_GDELT.ipynb Part 5, then re-run this notebook.")
    if REQUIRE_FULL_COVERAGE:
        raise SystemExit(msg + "\n\nTo score only what you hold, set "
                               "REQUIRE_FULL_COVERAGE = False.")
    print(msg)
    print("\nREQUIRE_FULL_COVERAGE is False -> scoring the held companies only. "
          "The lead list will NOT cover the client's full request.")
else:
    print("\nfull coverage - every company on the client list is held")

on the client list      : 92
already in companies.csv: 92
NOT held yet            : 0

full coverage - every company on the client list is held


## 3 · Features as of today

**Copied from `3_Flat_table.ipynb` section 4, unchanged.** The only differences are that `ASOF`
is today and no label is attached. Any edit here must be mirrored there, and vice versa.

Two scoring-time guards that the training build does not need:

- **Non-SME companies are not scored.** The model was trained on SMEs; a large company is out of
  distribution, so it is reported rather than given a misleading score.
- **Companies never enriched by Stage 3/4 are not scored.** Their charge columns would compute
  to 0, which reads as "has never borrowed" when the truth is "we have not looked yet" — the
  single most dangerous silent error in this notebook.

In [4]:
charges = pd.read_csv(CHARGES_CSV, dtype=str)
charges["com_num"]      = charges["com_num"].astype(str).str.strip().str.upper().str.zfill(8)
charges["created_on"]   = pd.to_datetime(charges["created_on"], errors="coerce")
charges["satisfied_on"] = pd.to_datetime(charges["satisfied_on"], errors="coerce")
charges["is_lloyds"]    = charges["is_lloyds"].astype(str).str.lower().eq("true")
charges = charges.dropna(subset=["created_on"])

# point-in-time cut (all charges filed on or before ASOF) + non-Lloyds only for features
nonlloyds = charges[(charges["created_on"] <= ASOF) & (~charges["is_lloyds"])]

d = companies[companies["com_num"].isin(set(client["com_num"]))].copy()
d = d.merge(client[["com_num", "relationship"]], on="com_num", how="left")
d["born"] = pd.to_datetime(d["date_of_creation"], errors="coerce")

# --- scoring-time eligibility buckets (reported, not silently dropped) ---
d["is_sme_flag"]   = d["is_sme"].astype(str).str.lower().eq("true")
d["never_enriched"] = d["recent_charge_status"].isna()

# ============ FEATURES — identical to 3_Flat_table.ipynb section 4 ============
g = nonlloyds.groupby("com_num")

d["nonlloyds_charges"]    = d["com_num"].map(g.size()).fillna(0).astype(int)
d["has_nonlloyds_charge"] = (d["nonlloyds_charges"] > 0).astype(int)
d["n_distinct_lenders"]   = d["com_num"].map(g["persons_entitled"].nunique()).fillna(0).astype(int)

_last = d["com_num"].map(g["created_on"].max())
d["yrs_since_nonlloyds_chg"] = ((ASOF - _last).dt.days / 365.25).fillna(50).clip(0, 50)

_recent = nonlloyds[nonlloyds["created_on"] >= ASOF - pd.DateOffset(months=24)]
d["n_charges_last_24m"] = d["com_num"].map(_recent.groupby("com_num").size()).fillna(0).astype(int)

_sat = nonlloyds[nonlloyds["satisfied_on"].notna() & (nonlloyds["satisfied_on"] <= ASOF)]
d["n_satisfied"]   = d["com_num"].map(_sat.groupby("com_num").size()).fillna(0).astype(int)
d["n_outstanding"] = (d["nonlloyds_charges"] - d["n_satisfied"]).clip(lower=0)

_lastsat = d["com_num"].map(_sat.groupby("com_num")["satisfied_on"].max())
d["yrs_since_satisfaction"] = ((ASOF - _lastsat).dt.days / 365.25).fillna(50).clip(0, 50)

d["age_years"] = (ASOF - d["born"]).dt.days / 365.25
d["accounts_overdue"] = d["accounts_overdue"].map(
    {"True": 1, True: 1, "False": 0, False: 0}).fillna(0).astype(int)

SEC = [(1,3,"A: Agriculture"),(5,9,"B: Mining"),(10,33,"C: Manufacturing"),(35,35,"D: Utilities"),
       (36,39,"E: Water/Waste"),(41,43,"F: Construction"),(45,47,"G: Retail/Wholesale"),
       (49,53,"H: Transport"),(55,56,"I: Accommodation/Food"),(58,63,"J: Information/Comms"),
       (64,66,"K: Finance/Insurance"),(68,68,"L: Real Estate"),(69,75,"M: Professional/Scientific"),
       (77,82,"N: Admin Support"),(84,84,"O: Public Admin"),(85,85,"P: Education"),
       (86,88,"Q: Health/Social"),(90,93,"R: Arts/Recreation"),(94,96,"S: Other Services"),
       (97,98,"T: Household Activities"),(99,99,"U: Extraterritorial")]

def section(code):
    """SIC code -> Companies House SIC section. 98 (property management) -> Real Estate."""
    try:
        div = int(str(code).strip()[:2])
    except (ValueError, TypeError):
        return None
    if div == 98:
        return "L: Real Estate"
    for lo, hi, s in SEC:
        if lo <= div <= hi:
            return s
    return None

d["sector"] = d["sic_code"].map(section)
d["region"] = d["region"].fillna("unknown") if "region" in d.columns else "unknown"
# =============================================================================

print(f"companies matched   : {len(d):,}")
print(f"  non-SME           : {int((~d['is_sme_flag']).sum()):,}  (not scored)")
print(f"  never enriched    : {int(d['never_enriched'].sum()):,}  (not scored - charges unknown)")
print(f"  no sector mapping : {int(d['sector'].isna().sum()):,}  (not scored)")
print(f"  no age            : {int(d['born'].isna().sum()):,}  (not scored)")

companies matched   : 92
  non-SME           : 2  (not scored)
  never enriched    : 2  (not scored - charges unknown)
  no sector mapping : 0  (not scored)
  no age            : 0  (not scored)


## 4 · Score

The manifest written by `4_model.ipynb` records the exact feature list **and its order**. The
assertion below is the point of that manifest: `OneHotEncoder(handle_unknown="ignore")` means a
missing or renamed column produces an all-zero block rather than an error, so without this check
a schema drift would silently degrade every score instead of failing.

Scores are **ranking values, not calibrated probabilities** — `class_weight="balanced"` inflates
them deliberately. Report a rank, never "a 39% chance of borrowing".

In [5]:
import joblib

model    = joblib.load(MODEL_FILE)
manifest = json.loads(MODEL_MANIFEST.read_text())
FEATURES = manifest["feature_cols"]

eligible = d[d["is_sme_flag"] & ~d["never_enriched"]
             & d["sector"].notna() & d["born"].notna()].copy()

missing_cols = [c for c in FEATURES if c not in eligible.columns]
assert not missing_cols, f"features missing from the scoring table: {missing_cols}"
X = eligible[FEATURES]                       # exact order from the manifest
assert list(X.columns) == FEATURES, "feature ORDER differs from the trained model"
assert not X.isna().any().any(), f"NaNs in: {X.columns[X.isna().any()].tolist()}"

eligible["score"] = model.predict_proba(X)[:, 1]

print(f"model    : {manifest['model']} trained {manifest['trained_at']}")
print(f"          on {manifest['trained_on_rows']:,} rows / {manifest['trained_on_positives']:,} positives")
print(f"scored   : {len(eligible):,} companies")
print(f"score range: {eligible['score'].min():.5f} - {eligible['score'].max():.5f}")

model    : logreg trained 2026-08-23T16:43:42
          on 360,600 rows / 1,203 positives
scored   : 90 companies
score range: 0.06427 - 0.76422


## 5 · Exclude existing customers, and reconcile

The client's `relationship` column and Companies House charge data are two independent views of
"is this already a customer". **The client's list wins** — they know their own book, including
the unsecured lending that never appears as a registered charge.

Where they disagree, report it rather than swallowing it. A company the client calls a prospect
while Companies House shows a Lloyds charge against it is worth a look: either their CRM extract
is stale, or it is a lapsed relationship — which is a lead, just a differently-framed one.

In [6]:
customers = set(client.loc[client["relationship"].eq("customer"), "com_num"])
leads = eligible[~eligible["com_num"].isin(customers)].copy()

# --- reconciliation: CH says Lloyds customer, the client says prospect ---
lloyds_ever = set(charges.loc[charges["is_lloyds"], "com_num"])
disagree = leads[leads["com_num"].isin(lloyds_ever)]

print(f"scored                 : {len(eligible):,}")
print(f"excluded as customers  : {len(eligible) - len(leads):,}")
print(f"remaining leads        : {len(leads):,}")
print(f"\ndisagreements (client says prospect, CH shows a Lloyds charge): {len(disagree):,}")
if len(disagree):
    print(disagree[["com_num", "name", "sector"]].head(10).to_string(index=False))

scored                 : 90
excluded as customers  : 0
remaining leads        : 90

disagreements (client says prospect, CH shows a Lloyds charge): 0


## 6 · Soft re-rank on sector × region media

A **tie-breaker, not a filter** — nothing is dropped, only the order shifts, and `GATE_WEIGHT`
bounds how far. Both scores and media are converted to percentile ranks first so the blend is
scale-free.

Only cells with genuine region-specific data (`z_source == "cell"`) are used. Roughly half the
index is `sector-fallback`, where a thin cell simply repeats the sector value — and on those
rows the measured association runs the *wrong way*, so they are given a neutral 0.5 rank rather
than being blended in. Yorkshire has no media cells at all and is likewise neutral.

In [7]:
leads["gated_rank"] = leads["score"].rank(pct=True)      # default if the gate is off

if USE_GATE and GATE_WEIGHT > 0 and MEDIA_INDEX.exists():
    mi = pd.read_parquet(MEDIA_INDEX)
    mi["week"] = pd.to_datetime(mi["week"])
    cells = mi[mi["z_source"].eq("cell")]                # real region data only
    prior = cells.loc[cells["week"] <= ASOF, "week"]
    if prior.empty:
        print(f"media index has no week on/before {ASOF.date()} -> gate skipped")
    else:
        use_week = prior.max()
        lut = (cells[cells["week"] == use_week]
               .set_index(["sector", "region"])["vol_z"])
        leads["vol_z"] = pd.MultiIndex.from_frame(
            leads[["sector", "region"]]).map(lut)

        r_score = leads["score"].rank(pct=True)
        # neutral 0.5 where there is no real cell -> neither boosted nor buried
        r_media = leads["vol_z"].rank(pct=True).fillna(0.5)
        leads["gated_rank"] = (1 - GATE_WEIGHT) * r_score + GATE_WEIGHT * r_media

        print(f"gate week {use_week.date()} (weight {GATE_WEIGHT})")
        print(f"  leads with a real media cell : {leads['vol_z'].notna().sum():,} "
              f"of {len(leads):,}")
        moved = (leads['score'].rank(ascending=False)
                 - leads['gated_rank'].rank(ascending=False)).abs()
        print(f"  median rank movement         : {moved.median():.0f} places")
else:
    print("gate disabled -> ranking on model score alone")

leads = leads.sort_values("gated_rank", ascending=False).reset_index(drop=True)
leads["rank"] = np.arange(1, len(leads) + 1)

gate week 2026-07-19 (weight 0.15)
  leads with a real media cell : 48 of 90
  median rank movement         : 2 places


## 7 · Why each lead — reason codes

An RM will not act on `0.0071`. For the logistic-regression model each feature's contribution is
`coefficient × standardised value`, which is directly readable: the three largest positive
contributions are why this company surfaced.

This is the practical payoff of keeping the primary model explainable. Skipped automatically if
a non-linear comparator was shipped instead.

In [8]:
def top_reasons(pipe, X_df, feature_names, n=3):
    """Top-n positive contributions per row for a linear model. None if not linear."""
    clf  = pipe.steps[-1][1]
    prep = pipe.steps[0][1]
    if not hasattr(clf, "coef_"):
        return None
    Z     = prep.transform(X_df)
    Z     = Z.toarray() if hasattr(Z, "toarray") else np.asarray(Z)
    names = np.asarray(prep.get_feature_names_out())
    contrib = Z * clf.coef_[0]
    idx = np.argsort(-contrib, axis=1)[:, :n]
    clean = lambda s: s.split("__", 1)[-1].replace("_", " ")
    return ["; ".join(clean(names[j]) for j in row) for row in idx]

reasons = top_reasons(model, leads[FEATURES], FEATURES)
if reasons is None:
    leads["why"] = ""
    print(f"'{manifest['model']}' is not linear -> reason codes skipped (use SHAP instead)")
else:
    leads["why"] = reasons
    print(leads[["com_num", "name", "score", "why"]].head(5).to_string(index=False))

 com_num                       name    score                                                                               why
08185784 1 CIRENCESTER ROAD LIMITED 0.764220                 yrs since nonlloyds chg; sector L: Real Estate; region North West
09336405                        NaN 0.756247 yrs since nonlloyds chg; sector L: Real Estate; account type total-exemption-full
SC663873                        NaN 0.654246                     region Scotland; account type total-exemption-full; age years
12328135                        NaN 0.720793     yrs since nonlloyds chg; account type total-exemption-full; region South East
15773489             116LRR LIMITED 0.671785  yrs since nonlloyds chg; region East Midlands; account type total-exemption-full


## 8 · Write the outputs

Three files per run, in a **dated** folder so history is preserved. Keeping every input and
output is what later makes it possible to ask *"did the leads we sent in Q3 actually convert?"* —
compare this run's list against the client's customer list next quarter. Overwrite the folder
and that measurement is gone permanently.

`not_found.csv` matters as much as `leads.csv`: handing back "here are your leads, and here are
the 43 numbers we could not match" is what makes the first file trustworthy.

**Output quality gate.** Section 4 validates the model's *input* schema; this validates the
*output*. Nothing previously checked the columns handed to the client, which is exactly how a batch
of leads once went out with a blank `name` — it is not a model feature, so no assertion touched it
and it failed silently until someone opened the file. The gate refuses to write `leads.csv` at all
rather than write one with holes in it.

In [9]:
# 1. the ranked lead list
OUT_COLS = ["rank", "com_num", "name", "sector", "region", "account_type",
            "age_years", "nonlloyds_charges", "n_outstanding", "n_satisfied",
            "score", "why"]
top = leads.head(TOP_K)[[c for c in OUT_COLS if c in leads.columns]]

# --- output quality gate: never write a deliverable with holes in it -------------
# The model's INPUT schema is checked in section 4, but nothing used to check the
# OUTPUT. That is how 48 leads went out with a blank `name`: the column is not a
# feature, so no assertion touched it and it failed silently until someone read
# the file. The columns nothing checks are the ones that break quietly.
OPTIONAL_COLS = {"why"}          # legitimately empty if a non-linear model shipped
_holes = {c: int(top[c].isna().sum()) for c in top.columns
          if c not in OPTIONAL_COLS and top[c].isna().any()}
if _holes:
    _detail = ", ".join(f"'{c}' missing in {n} of {len(top)} rows"
                        for c, n in _holes.items())
    raise ValueError(
        f"leads.csv NOT written - incomplete columns: {_detail}.\n"
        "A blank `name` usually means those companies entered via a client list "
        "before `name` was part of the Stage 1 profile pull.\n"
        "Fix: re-run Stage 1 in 1_CompaniesHouse.ipynb (its `todo` self-heals rows "
        "missing a name), then re-run this notebook."
    )
_checked = [c for c in top.columns if c not in OPTIONAL_COLS]
print(f"output quality: all {len(_checked)} required columns complete across {len(top):,} rows")

top.to_csv(RUN_DIR / "leads.csv", index=False)

# 2. everything we could not score, with the reason -- nothing vanishes silently
unscored = []
for num in sorted(set(client["com_num"]) - set(d["com_num"])):
    unscored.append((num, "not found in Companies House"))
for _, r in d.iterrows():
    if   not r["is_sme_flag"]:        why = "not an SME (out of model scope)"
    elif r["never_enriched"]:         why = "charge history not pulled yet - run Stages 3/4"
    elif pd.isna(r["sector"]):        why = "SIC code could not be mapped to a sector"
    elif pd.isna(r["born"]):          why = "no incorporation date"
    elif r["com_num"] in customers:   why = "existing customer - excluded from leads"
    else:                             continue
    unscored.append((r["com_num"], why))
not_found = pd.DataFrame(unscored, columns=["com_num", "reason"])
not_found.to_csv(RUN_DIR / "not_found.csv", index=False)

# 3. the run manifest -- what a score can be traced back to
(RUN_DIR / "run_manifest.txt").write_text("\n".join([
    f"run date        : {dt.date.today().isoformat()}",
    f"ASOF            : {ASOF.date()}",
    f"client files    : {', '.join(f.name for f in CLIENT_FILES)}",
    f"rows received   : {n_raw:,}   unique: {len(client):,}",
    f"model           : {manifest['model']}  trained {manifest['trained_at']}",
    f"model source    : {manifest['source_table']} (sha {manifest['source_sha256']})",
    f"companies.csv   : {len(companies):,} rows",
    f"coverage        : {len(client) - len(missing):,} of {len(client):,} companies held",
    f"gate            : {'on, weight ' + str(GATE_WEIGHT) if USE_GATE else 'off'}",
    f"scored          : {len(eligible):,}",
    f"excluded (cust) : {len(eligible) - len(leads):,}",
    f"leads written   : {len(top):,} of {len(leads):,}",
    f"not scored      : {len(not_found):,}",
]) + "\n")

print((RUN_DIR / "run_manifest.txt").read_text())
print(f"-> {RUN_DIR}/leads.csv, not_found.csv, run_manifest.txt")
top.head(10)

run date        : 2026-08-23
ASOF            : 2026-08-23
client files    : 30k_sme_for_mock_up.csv
rows received   : 99   unique: 92
model           : logreg  trained 2026-08-23T16:43:42
model source    : /Users/natchalin_/Projects/final_project/Lloyds/API/flat_pot.csv (sha 9c1daa03ba2e745b)
companies.csv   : 839,344 rows
coverage        : 92 of 92 companies held
gate            : on, weight 0.15
scored          : 90
excluded (cust) : 0
leads written   : 90 of 90
not scored      : 2

-> /Users/natchalin_/Projects/final_project/Lloyds/client/output/2026-08-23/leads.csv, not_found.csv, run_manifest.txt


,rank,com_num,name,sector,region,account_type,age_years,nonlloyds_charges,n_outstanding,n_satisfied,score,why
0,1,08185784,1 CIRENCESTER ROAD LIMITED,L: Real Estate,North West,total-exemption-full,14.004107,2,2,0,0.764220,yrs since nonlloyds chg; sector L: Real Estate...
1,2,09336405,NaN,L: Real Estate,East of England,total-exemption-full,11.723477,1,1,0,0.756247,yrs since nonlloyds chg; sector L: Real Estate...
2,3,SC663873,NaN,F: Construction,Scotland,total-exemption-full,6.198494,0,0,0,0.654246,region Scotland; account type total-exemption-...
3,4,12328135,NaN,I: Accommodation/Food,South East,total-exemption-full,6.751540,1,1,0,0.720793,yrs since nonlloyds chg; account type total-ex...
4,5,15773489,116LRR LIMITED,F: Construction,East Midlands,total-exemption-full,2.195756,5,5,0,0.671785,yrs since nonlloyds chg; region East Midlands;...
5,6,13407380,11 TURNER STREET LIMITED,L: Real Estate,London,total-exemption-full,5.264887,1,1,0,0.758398,yrs since nonlloyds chg; sector L: Real Estate...
6,7,02339455,02339455 LIMITED,G: Retail/Wholesale,Yorkshire,small,37.571526,0,0,0,0.641447,account type small; sector G: Retail/Wholesale...
7,8,12875796,NaN,L: Real Estate,Yorkshire,total-exemption-full,5.941136,0,0,0,0.625246,sector L: Real Estate; account type total-exem...
8,9,SC457356,0898 CONSULTING LIMITED,M: Professional/Scientific,Scotland,total-exemption-full,13.004791,0,0,0,0.521073,region Scotland; account type total-exemption-...
9,10,12733091,10STAR SECURITY SOLUTIONS LTD,N: Admin Support,West Midlands,total-exemption-full,6.119097,0,0,0,0.602538,region West Midlands; account type total-exemp...


## 9 · Checks

**Conservation is the one that catches real bugs.** Every company the client sent must end up in
exactly one bucket — scored, excluded, or reported as unscoreable. A silent drop of forty
companies somewhere between input and output is the classic failure of a pipeline like this, and
plain arithmetic is what catches it.

In [10]:
unique_in  = len(client)
accounted  = len(leads) + len(not_found)
print(f"unique companies in   : {unique_in:,}")
print(f"leads out             : {len(leads):,}")
print(f"reported as unscored  : {len(not_found):,}")
print(f"accounted for         : {accounted:,}")
assert accounted == unique_in, f"CONSERVATION FAILED: {unique_in - accounted} companies vanished"

assert not set(leads["com_num"]) & customers, "an existing customer reached the lead list"
assert leads["score"].between(0, 1).all(), "scores outside [0, 1]"
assert leads["rank"].is_monotonic_increasing and leads["rank"].iloc[0] == 1, "ranking is broken"
assert not_found["com_num"].is_unique, "a company is reported unscored twice"
print("\nall checks passed")

unique companies in   : 92
leads out             : 90
reported as unscored  : 2
accounted for         : 92

all checks passed
